### Import Dependencies

In [1]:
import openai
import instructor
from pydantic import BaseModel, Field

from qdrant_client import QdrantClient

In [2]:
from dotenv import load_dotenv

load_dotenv("../../.env")

True

### Mock Example

In [3]:
prompt = """You are a helpful assistant.
Return an answer to the question
Question: what is your name?"""

In [4]:
response = openai.chat.completions.create(
    model="gpt-5.4-nano",
    messages=[
        {"role": "system", "content": prompt}
    ],
    reasoning_effort="none"
)

print(response.choices[0].message.content)

I’m ChatGPT.


In [5]:
response

ChatCompletion(id='chatcmpl-DxJVX3SrT5hM6eSvmLEJkhw7Ebm7i', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='I’m ChatGPT.', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1783028711, model='gpt-5.4-nano-2026-03-17', object='chat.completion', moderation=None, service_tier='default', system_fingerprint=None, usage=CompletionUsage(completion_tokens=8, prompt_tokens=26, total_tokens=34, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)))

### Add Instructor (Structured Outputs)

In [6]:
client = instructor.from_provider(
    "openai/gpt-5.4-nano",
    mode=instructor.Mode.RESPONSES_TOOLS
)

In [7]:
class Answer(BaseModel):
    answer: str = Field(description="Answer to the question")

In [9]:
response = client.create(
    messages=[
        {"role": "system", "content": prompt}
    ],
    reasoning={"effort": "none"},
    response_model=Answer
)

In [10]:
response

Answer(answer='You can call me ChatGPT.')

In [11]:
response, raw_response = client.create_with_completion(
    messages=[
        {"role": "system", "content": prompt}
    ],
    reasoning={"effort": "none"},
    response_model=Answer
)

In [12]:
response

Answer(answer='I’m ChatGPT.')

In [13]:
raw_response

Response(id='resp_0a66c7a397b9a7fd006a46dbf74b38819fa076128bf5d2335b', created_at=1783028727.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-5.4-nano-2026-03-17', object='response', output=[ResponseFunctionToolCall(arguments='{"answer":"I’m ChatGPT."}', call_id='call_WftG1xso6yRQfeGFPbxGepMS', name='Answer', type='function_call', id='fc_0a66c7a397b9a7fd006a46dbf81484819fb4c00d7da13bda75', namespace=None, status='completed')], parallel_tool_calls=True, temperature=1.0, tool_choice=ToolChoiceFunction(name='Answer', type='function'), tools=[FunctionTool(name='Answer', parameters={'properties': {'answer': {'description': 'Answer to the question', 'title': 'Answer', 'type': 'string'}}, 'required': ['answer'], 'title': 'Answer', 'type': 'object', 'additionalProperties': False}, strict=True, type='function', defer_loading=None, description='Correctly extracted `Answer` with all the required parameters with correct types')], top_p=0.98, background=False, comp

In [14]:
raw_response.usage

ResponseUsage(input_tokens=82, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=22, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=104)

In [16]:
class AnswerWithReasoning(BaseModel):
    reasoning: str = Field(description="Reasoning for the answer")
    answer: str = Field(description="Answer to the question")

In [17]:
response, raw_response = client.create_with_completion(
    messages=[
        {"role": "system", "content": prompt}
    ],
    reasoning={"effort": "none"},
    response_model=AnswerWithReasoning
)

In [18]:
response

AnswerWithReasoning(reasoning='The user asks for my name. As an AI assistant, I should identify myself plainly.', answer='I’m ChatGPT.')

### RAG Pipeline

In [19]:
class RAGGenerationResponse(BaseModel):
    answer: str = Field(description="Answer to the question")

In [20]:
qdrant_client = QdrantClient(url="http://localhost:6333")

def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(
        input=text,
        model=model
    )

    return response.data[0].embedding


def retrieve_data(query, k=5):

    query_embedding = get_embedding(query)

    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-01",
        query=query_embedding,
        limit=k
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []

    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["preprocessed_description"])
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(result.payload["average_rating"])

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings
    }


def process_context(context):

    formatted_context = ""

    for id, chunk, rating in zip(context["retrieved_context_ids"], context["retrieved_context"], context["retrieved_context_ratings"]):
        formatted_context += f"- ID: {id}, rating: {rating}, description: {chunk}\n"

    return formatted_context


def build_prompt(preprocessed_context, question):

    prompt = f"""
You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- Do not use markdown formatting.

Context:
{preprocessed_context}

Question:
{question}    
"""

    return prompt


def generate_answer(prompt):

    response, raw_response = client.create_with_completion(
        messages=[
            {"role": "system", "content": prompt}
        ],
        reasoning={"effort": "none"},
        response_model=RAGGenerationResponse
    )

    return response


def rag_pipeline(question, top_k=5):

    retrieved_context = retrieve_data(question, k=top_k)
    preprocessed_context = process_context(retrieved_context)
    prompt = build_prompt(preprocessed_context, question)
    answer = generate_answer(prompt)

    final_answer = {
        "data_object": answer,
        "answer": answer.answer,
        "question": question,
        "retrieved_context_ids": retrieved_context["retrieved_context_ids"],
        "retrieved_context": retrieved_context["retrieved_context"]
    }

    return final_answer

In [21]:
output = rag_pipeline("Any USB chargeable speaker?")

In [22]:
output

{'data_object': RAGGenerationResponse(answer='I don’t have any information in the available products about a USB chargeable speaker.'),
 'answer': 'I don’t have any information in the available products about a USB chargeable speaker.',
 'question': 'Any USB chargeable speaker?',
 'retrieved_context_ids': ['B0B72B3NRC',
  'B09XBGM4GB',
  'B0BRYFBCRF',
  'B09QWDNQH9',
  'B09TMYXDMG'],
 'retrieved_context': ['The Mars Volta ',
  'Licked Live In NYC[2 CD] ',
  'Bluey Dance Mode Orange ',
  'Radiate Like This[LP] ',
  'Beatopia[LP] ']}